# Lab 5-1: Introduction to Spark

**Lab Description:**  
This lab introduces Spark fundamentals in Databricks, focusing on data ingestion, exploration, and feature engineering. You will create catalogs, schemas, and volumes, upload CSV files, load them into Spark DataFrames, inspect data, compute features, and explore Spark's lazy evaluation and logical plan.

**Topics and Commands Covered:**
- Catalog, schema, and volume creation (`CREATE CATALOG`, `CREATE SCHEMA`, `CREATE VOLUME`)
- File upload and managed storage in Databricks
- Loading CSV files into DataFrames (`spark.read.csv`)
- DataFrame inspection (`display`, `.count()`)
- DataFrame lineage and logical plan (`.explain()`)
- Feature engineering with aggregations
- Lazy evaluation in Spark

Data files used:

- users.csv: ~20 users. dimension table
- events.csv: ~120 rows. fact table

## What’s in the files

### `users.csv`

 column        | description                                    |
 ------------- | ---------------------------------------------- |
 `user_id`     | integer user identifier                        |
 `state`       | small categorical dimension (CA, NY, TX, etc.) |
 `signup_date` | date (string → good for casting demo)          |


### `events.csv`

 column       | description                       |
 ------------ | --------------------------------- |
 `event_id`   | unique event id                   |
 `user_id`    | includes some NULLs + invalid IDs |
 `event_ts`   | timestamp string                  |
 `event_type` | view / click / purchase           |
 `amount`     | numeric with NULLs                |

### 1\. Create Catalog, Schema, and Volume for this lab

In Databricks, a **catalog** organizes collections of schemas, while a **schema** (also called a database) groups related tables and views. A **volume** is a managed storage location within a schema, used for storing files and data assets.

Uses:
- **Catalog**: Segregates data for governance and access control.
- **Schema**: Structures data for logical organization and querying.
- **Volume**: Stores raw files (e.g., CSVs) for ingestion and processing.

In this lab, we create a volume to upload and manage data files (like `users.csv` and `events.csv`) for Spark processing and analysis.

You may do so using the following SparkSQL statements:

```sql

CREATE CATALOG IF NOT EXISTS msbabigdata;
CREATE SCHEMA IF NOT EXISTS msbabigdata.spark;
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.lab5_1;
```

In [ ]:
%sql

CREATE CATALOG IF NOT EXISTS msbabigdata;
CREATE SCHEMA IF NOT EXISTS msbabigdata.spark;
CREATE VOLUME IF NOT EXISTS msbabigdata.spark.lab5_1;

### 2\. Upload the data file into the volume for this lab

To upload `users.csv` and `events.csv` into the volume using Databricks UI:

1. In the left sidebar, click **Data**.
2. Navigate to the catalog: `msbabigdata`, then schema: `spark`, then volume: `lab5_1`.
3. Click the **Upload** button.
4. Select `users.csv` and `events.csv` from your local machine and upload them.
5. Confirm the files appear in the volume directory.

You can now access these files at `/Volumes/msbabigdata/spark/lab5_1/users.csv` and `/Volumes/msbabigdata/spark/lab5_1/events.csv`.

### 3\. Verify Spark Session `spark`

A **Spark session** is the entry point to using Spark for data processing. It manages resources, configuration, and allows you to create DataFrames, run SQL queries, and interact with Spark features. In Databricks, the `spark` session variable is created automatically for you.

Verify the Spark session by running the code below.

In [ ]:
#in jupiter notebook we have a combination of python and spark tehn how would you differentiate between the two
# just typr spark to create spark domain
spark

### 4\. Load CSV files into Spark DataFrames

To load CSV files into Spark DataFrames, follow these steps:

**Step 1: Determine the file paths**

The files are located in the Databricks volume you created (you may use "Copy path" through Catalog explorer UI.):
- `users.csv`: `/Volumes/msbabigdata/spark/lab5_1/users.csv`
- `events.csv`: `/Volumes/msbabigdata/spark/lab5_1/events.csv`

**Step 2: Use `spark.read.csv` with options**

- `header=True`: Tells Spark that the first row contains column names.
- `inferSchema=True`: Tells Spark to automatically detect data types for each column.

**Example: Load `users.csv` into a DataFrame called `users`**

---

```python
users_path = "/Volumes/msbabigdata/spark/lab5_1/users.csv"
users = spark.read.csv(users_path, header=True, inferSchema=True)
```

> Note that we used spark session as a starting point for creating Spark DataFrames

Follow this example to create two Spark DataFrames (`users`, `events`) for analysis.

In [ ]:
#to read the csv data from this path
users_path = "/Volumes/msbabigdata/spark/lab5_1/users.csv"
users = spark.read.csv(users_path, header=True, inferSchema=True)

events_path = "/Volumes/msbabigdata/spark/lab5_1/events.csv"
events = spark.read.csv(events_path, header=True, inferSchema=True)

In [ ]:
# it doesnt show you the content because this is lazy
users

### 5\. Inspect the Spark DataFrames

To display some rows of each Spark DataFrame, use the `display()` method:

```python
# Display first ~1000 rows of users DataFrame
users.diplay()
```

Follow the example to display sample rows from each DataFrame

In [ ]:
# doesnt give all bbteh data as your computer may not even hold that big amt of data
users.display()

user_id | state | signup_date
1 | WA | 2022-10-03
2 | TX | 2022-07-01
3 | TX | 2022-07-21
4 | WA | 2022-02-22
5 | MN | 2022-03-24
6 | CA | 2022-02-08
7 | MN | 2022-09-29
8 | MN | 2022-12-12
9 | NY | 2022-05-03
10 | NY | 2022-01-28
11 | WA | 2022-01-11
12 | NY | 2022-04-07
13 | TX | 2022-02-03
14 | CA | 2022-01-01
15 | IL | 2022-04-27
16 | IL | 2022-12-12
17 | NY | 2022-02-16
18 | IL | 2022-09-20
19 | TX | 2022-09-19
20 | WA | 2022-07-16

In [ ]:
events.display()

event_id | user_id | event_ts | event_type | amount
1 | 1 | 7/7/2022 14:00 | click | 155.87
2 | 21 | 11/4/2022 6:00 | view | 169.97
3 | 19 | 6/19/2022 0:00 | view | 150.79
4 | 22 | 9/24/2022 19:00 | purchase | None
5 | 11 | 1/12/2022 20:00 | view | None
6 | 3 | 4/19/2022 17:00 | view | None
7 | 4 | 2/13/2022 14:00 | purchase | 147.81
8 | 5 | 2/8/2022 15:00 | click | None
9 | 2 | 3/2/2022 12:00 | view | 110.02
10 | 10 | 10/4/2022 11:00 | view | 18.66
11 | 10 | 1/18/2022 14:00 | click | 151.39
12 | 10 | 6/9/2022 0:00 | view | None
13 | 17 | 10/11/2022 13:00 | click | 35.26
14 | 16 | 8/19/2022 13:00 | click | None
15 | 16 | 8/20/2022 15:00 | view | 181.42
16 | 1 | 3/31/2022 13:00 | click | None
17 | 4 | 12/18/2022 15:00 | purchase | None
18 | 14 | 11/24/2022 5:00 | view | 80.02
19 | 17 | 6/5/2022 1:00 | purchase | None
20 | 22 | 7/11/2022 21:00 | click | 72.51
... (120 rows total)

### 6\. How many records are there in the `events` Spark DataFrame (use the `count()` method)

In [ ]:
%python
__backend_agg_display_orig = display
__backend_agg_dfs = []
def __backend_agg_display_new(df):
    __backend_agg_df_modules = ["pandas.core.frame", "databricks.koalas.frame", "pyspark.sql.dataframe", "pyspark.pandas.frame", "pyspark.sql.connect.dataframe"]
    if (type(df).__module__ in __backend_agg_df_modules and type(df).__name__ == 'DataFrame') or isinstance(df, list):
        __backend_agg_dfs.append(df)

display = __backend_agg_display_new

def __backend_agg_user_code_fn():
    import base64
    exec(base64.standard_b64decode("dXNlcnMuZGlzcGxheSgp").decode())

try:
    # run user code
    __backend_agg_user_code_fn()

    #reset display function
    display = __backend_agg_display_orig

    if len(__backend_agg_dfs) > 0:
        # create a temp view
        if type(__backend_agg_dfs[0]).__module__ == "databricks.koalas.frame":
            # koalas dataframe
            __backend_agg_dfs[0].to_spark().createOrReplaceTempView("DatabricksView0ab21c6")
        elif type(__backend_agg_dfs[0]).__module__ == "pandas.core.frame" or isinstance(__backend_agg_dfs[0], list):
            # pandas dataframe
            spark.createDataFrame(__backend_agg_dfs[0]).createOrReplaceTempView("DatabricksView0ab21c6")
        else:
            __backend_agg_dfs[0].createOrReplaceTempView("DatabricksView0ab21c6")
        #run backend agg
        display(spark.sql("""WITH q AS (select * from DatabricksView0ab21c6) SELECT `signup_date`,`state`,SUM(`user_id`) `column_ef5e8e58122` FROM q GROUP BY `signup_date`,`state`"""))
    else:
        displayHTML("dataframe no longer exists. If you're using dataframe.display(), use display(dataframe) instead.")


finally:
    spark.sql("drop view if exists DatabricksView0ab21c6")
    display = __backend_agg_display_orig
    del __backend_agg_display_new
    del __backend_agg_display_orig
    del __backend_agg_dfs
    del __backend_agg_user_code_fn



In [ ]:

users.count()

In [ ]:
events.count()

### 7\. Inspect the `events` DataFrame's lineage

A DataFrame's lineage shows the sequence of transformations and logical plan used to produce it. Although, we cannot directly access a DataFrame's lineage, we can get something close to it by calling the `.explain()` method---which describes the steps Sparks will take to obtain this DataFrame.

To inspect the "lineage" of the `events` DataFrame, run:


```python
events.explain()
```

In [ ]:
#we do not have exactly lineage, but we can get something like that
events.explain()

== Physical Plan ==
FileScan csv [event_id#13238,user_id#13239,event_ts#13240,event_type#13241,amount#13242] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/msbabigdata/spark/lab5_1/events.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<event_id:int,user_id:int,event_ts:string,event_type:string,amount:double>




### 8\. Compute additional features for users from the `events` dataset.

In this step, we identify and extract relevant columns from the `events` DataFrames to use as features for a machine learning pipeline. 

Run the code below to compute candidate features from your DataFrames, including:

- **num_events**: Number of events associated with each user (captures user activity level).
- **total_amount**: Sum of the `amount` field for each user (captures total spending or value).
- **avg_amount**: Average of the `amount` field for each user (captures typical event value per user).

In [ ]:
#we need to do some predictive modelling at the user level


In [ ]:
# Import aggregation functions from pyspark.sql.functions
from pyspark.sql.functions import count, sum, avg

# Group the events DataFrame by 'user_id' and aggregate:
# - count(*) to get the number of events per user
# - sum('amount') to get the total amount per user
# - avg('amount') to get the average amount per user

# we create a new dataframe called features
features = events.groupBy("user_id").agg(
    count("*").alias("num_events"),      # Count number of events for each user
    sum("amount").alias("total_amount"), # Sum of 'amount' for each user
    avg("amount").alias("avg_amount")    # Average 'amount' for each user

)



### 9\. Inspects the `features` dataframe

**Spark is Lazily Evaluated**

When you type `features`, Spark only shows a summary of the DataFrame's schema and logical plan—it does **not** actually compute or retrieve any data. This is because Spark uses lazy evaluation: transformations are recorded but not executed until an action is triggered.

To actually see the data, you must run `features.display()`. This command triggers a Spark action, causing Spark to execute the necessary computations and return the results for display.

- `features`: Shows schema and plan, no data is computed yet.
- `features.display()`: Triggers computation and displays the data.

In [ ]:
features

# this does not give much result as it is the lazy transformation

In [ ]:
features.display()

user_id | num_events | total_amount | avg_amount
12 | 4 | 178.92 | 178.92
18 | 8 | 375.56 | 125.18666666666667
16 | 7 | 575.69 | 115.138
5 | 7 | 317.61 | 79.4025
10 | 4 | 250.51 | 83.50333333333333
1 | 6 | 610.4300000000001 | 152.60750000000002
3 | 6 | 377.37 | 94.3425
20 | 3 | 172.28 | 86.14
22 | 9 | 805.73 | 134.28833333333333
2 | 3 | 110.02 | 110.02
14 | 9 | 1083.16 | 135.395
13 | 4 | 368.11 | 184.055
17 | 9 | 354.47 | 59.07833333333334
6 | 4 | 484.16 | 121.04
9 | 4 | 348.15 | 87.0375
None | 4 | 136.65 | 68.325
11 | 5 | 108.47 | 36.156666666666666
7 | 3 | 161.81 | 80.905
19 | 5 | 523.61 | 130.9025
15 | 2 | 205.39 | 102.695
... (23 rows total)

### 10\. Use `explain` to explain `features`'s execution plan ('lineage').

> When you run `features.explain()`, Spark prints the logical and physical plan for computing the DataFrame.  
> - **Interpretation:** The output shows the sequence of transformations (e.g., groupBy, aggregations) Spark will perform, but no data is actually processed yet.  
> - **Why:** Spark uses lazy evaluation—transformations are recorded as a lineage (plan), and only executed when an action (like `display()` or `count()`) is triggered.  
> - **Lineage:** The explain output helps you understand how Spark will optimize and execute your computation, tracing back through all steps used to build the DataFrame.

In [ ]:
# remeber each fdataframe islzy

features.explain()

#shuffle - we need to do it at the time of word count

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   ColumnarToRow
   +- PhotonResultStage
      +- PhotonGroupingAgg(keys=[user_id#13239], functions=[finalmerge_count(merge count#13333L) AS count(1)#13327L, finalmerge_sum(merge sum#13335) AS sum(amount)#13328, finalmerge_count(merge count#13337L) AS count(amount)#13331L])
         +- PhotonShuffleExchangeSource
            +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#8645]
               +- PhotonShuffleExchangeSink hashpartitioning(user_id#13239, 16)
                  +- PhotonGroupingAgg(keys=[user_id#13239], functions=[partial_count(1) AS count#13333L, partial_sum(amount#13242) AS sum#13335, partial_count(amount#13242) AS count#13337L])
                     +- PhotonRowToColumnar
                        +- FileScan csv [user_id#13239,amount#13242] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[dbfs:/Volumes/msbabigdata/spark/lab5_1/events.csv], PartitionFilters: 

### 11\. Save (Persist) the results to the Utility catalog volume.

Spark DataFrames exist only in memory and are lost when your session ends. To keep results for later use, you must save them as data files (e.g., CSV, Parquet) in a persistent storage location.

Run the command below to save your DataFrame:

```python
features.write.mode("overwrite").csv("/Volumes/msbabigdata/spark/lab5_1/features")
```

In [ ]:

#to overwrite from where it aleady exists, save. it as csv
features.write.mode("overwrite").csv("/Volumes/msbabigdata/spark/lab5_1/features")

# what would happen after we run this
# lazy and we did not have the features in the csv, to csv aggregate and save it
# when we noe look at the catalogue we find a features foler there and when you open it you see multiple files created, but one is csv and that is the acatual file that we need
# We do not know what tthe other files are

### 12\. Inspect the output

Use **Catalog Explorer** to observe the output at `/Volumes/msbabigdata/spark/lab5_1/features`.  
Notice that the output path is treated as a **directory**, not a single file. This is because Spark writes data in parallel across multiple partitions, creating several files within the specified directory.  
When providing an output path, always specify a directory (not a file name), as Spark will manage the files inside it.  
You can explore the directory in Catalog Explorer to see the individual files produced by Spark.


### 13\. (optional) Clean up the lab volume

To remove the volume along with files or directories in it, use the SparkSQL `DROP VOLUME ` command. For example:

```sql
DROP VOLUME <catalog>.<schema>.<volume>;
```

Write your own command to clean up the lab volume `lab5_1`.

> note that this step is irreversable. Make sure you are certain about deleting the volume entirely. 



In [ ]:
%sql

DROP VOLUME msbabigdata.spark.lab5_1;


## This is the end of the lab